# Qubitization and Double Factorization

This notebook demonstrates qubitization -- an alternative to
Trotterization with better asymptotic scaling for chemistry
simulation. Covers double factorization, gate-level LCU
circuits, and the qsharp.chemistry bridge for production
resource estimation.

**Requirements:** `pip install qdk-pythonic[chemistry]`

## 1. Double Factorization

Double factorization decomposes the two-electron integrals into
a low-rank form, reducing the 1-norm (lambda) that determines
qubitization cost.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import run_scf, get_integrals
from qdk_pythonic.domains.common.double_factorization import double_factorize

# H2 molecule
scf = run_scf("H 0 0 0; H 0 0 0.74", basis="sto-3g")
h1e, h2e, nuc = get_integrals(scf)

df = double_factorize(h1e, h2e, nuc, n_electrons=2)
df.print_summary()

### Threshold Effect

A higher threshold truncates more eigenvalues, reducing rank
(and accuracy).

In [ ]:
for thresh in [1e-10, 1e-6, 1e-2, 0.1, 0.5]:
    df_t = double_factorize(h1e, h2e, nuc, n_electrons=2, threshold=thresh)
    print(f"Threshold {thresh:.0e}: {df_t.n_leaves} leaves, "
          f"1-norm = {df_t.one_norm():.4f}")

### Round-Trip Validation

The factorized form can reconstruct the original integrals.

In [ ]:
import numpy as np

df_precise = double_factorize(h1e, h2e, nuc, n_electrons=2, threshold=1e-12)
fcidump = df_precise.to_fcidump_data()
diff = np.max(np.abs(h2e - fcidump.h2e))
print(f"Reconstruction error: {diff:.2e}")

## 2. Gate-Level LCU Components

The LCU framework provides PREPARE, SELECT, and walk operator
circuits for qubitization.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_hamiltonian
from qdk_pythonic.domains.common.lcu import (
    PrepareOracle, SelectOracle, QubitizationWalkOperator,
)

h = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", basis="sto-3g")

print(f"Hamiltonian: {len(h)} terms on {h.qubit_count()} qubits")
print(f"1-norm: {h.summary()['one_norm']:.4f}")

# PREPARE oracle
prep = PrepareOracle(h)
print(f"\nPREPARE: {prep.n_ancilla_qubits} ancilla qubits")
prep_circ = prep.to_circuit()
print(f"  Circuit: {prep_circ.total_gate_count()} gates")

# SELECT oracle
sel = SelectOracle(h)
print(f"\nSELECT: {sel.n_system_qubits} system + {sel.n_ancilla_qubits} ancilla")
sel_circ = sel.to_circuit()
print(f"  Circuit: {sel_circ.total_gate_count()} gates")

# Walk operator
walk = QubitizationWalkOperator(h)
walk_circ = walk.to_circuit()
print(f"\nWalk operator: {walk_circ.qubit_count()} qubits, "
      f"{walk_circ.total_gate_count()} gates")

## 3. Qubitization QPE

QPE on the walk operator extracts energy eigenvalues.

In [ ]:
from qdk_pythonic.domains.common.lcu import QubitizationQPE

qpe = QubitizationQPE(hamiltonian=h, n_electrons=2, n_estimation_qubits=4)
circuit = qpe.to_circuit()

print(f"Qubitization QPE circuit:")
print(f"  Qubits: {circuit.qubit_count()}")
print(f"  Gates:  {circuit.total_gate_count()}")
print(f"  Depth:  {circuit.depth()}")

### Energy from Phase

The qubitization walk operator encodes energies as
`E = lambda * sin(2*pi*phi - pi/2)`.

In [ ]:
one_norm = h.summary()["one_norm"]

for phase in [0.0, 0.25, 0.5, 0.75]:
    energy = QubitizationQPE.energy_from_phase(phase, one_norm)
    print(f"Phase {phase:.2f} -> Energy {energy:.4f} Ha")

## 4. ChemistryQubitization Wrapper

The high-level wrapper supports both gate-level circuits and
the qsharp.chemistry bridge.

In [ ]:
from qdk_pythonic.domains.chemistry import ChemistryQubitization

# Gate-level mode (small systems)
cq = ChemistryQubitization(
    hamiltonian=h,
    n_electrons=2,
    n_estimation_qubits=4,
    gate_level=True,
)
circuit = cq.to_circuit()
print(f"Gate-level qubitization: {circuit.qubit_count()} qubits, "
      f"{circuit.total_gate_count()} gates")

# Bridge mode for production resource estimation:
# cq_prod = ChemistryQubitization(
#     hamiltonian=df,  # DoubleFactorizedHamiltonian
#     n_electrons=2,
#     gate_level=False,
# )
# result = cq_prod.estimate_resources()
# result.print_report()

## 5. Trotter vs Qubitization Comparison

Compare QPE circuit sizes between Trotter and qubitization.

In [ ]:
from qdk_pythonic.domains.chemistry import ChemistryQPE

# Trotter QPE
trotter_qpe = ChemistryQPE(
    hamiltonian=h, n_electrons=2,
    n_estimation_qubits=4, trotter_steps=2,
)
trotter_circ = trotter_qpe.to_circuit()

# Qubitization QPE
qubit_qpe = QubitizationQPE(
    hamiltonian=h, n_electrons=2, n_estimation_qubits=4,
)
qubit_circ = qubit_qpe.to_circuit()

print(f"{'Method':<20} {'Qubits':>8} {'Gates':>10} {'Depth':>8}")
print("-" * 50)
print(f"{'Trotter QPE':<20} {trotter_circ.qubit_count():>8} "
      f"{trotter_circ.total_gate_count():>10} {trotter_circ.depth():>8}")
print(f"{'Qubitization QPE':<20} {qubit_circ.qubit_count():>8} "
      f"{qubit_circ.total_gate_count():>10} {qubit_circ.depth():>8}")

## 6. Structured Resource Estimation

The `ChemistryResourceEstimate` wraps raw estimation output.

In [ ]:
from qdk_pythonic.execution.chemistry_estimate import (
    parse_estimation_result, compare_estimates,
)

# Example with mock data (real usage requires qsharp)
mock_result = {
    "logicalCounts": {"numQubits": 100, "tCount": 50000},
    "physicalCounts": {"physicalQubits": 200000, "runtime": 3600000000},
    "physicalCountsFormatted": {"runtime": "1 hour"},
    "logicalQubit": {"codeDistance": 17},
    "jobParams": {
        "qubitParams": {"name": "qubit_gate_ns_e3"},
        "qecScheme": {"name": "surface_code"},
        "errorBudget": 0.01,
    },
}

result = parse_estimation_result(
    mock_result,
    algorithm_name="trotter_qpe",
    hamiltonian_info={"n_orbitals": 2, "n_electrons": 2},
)
result.print_report()

### Comparison Table

In [ ]:
r1 = parse_estimation_result(mock_result, algorithm_name="trotter_qpe")
r2 = parse_estimation_result(mock_result, algorithm_name="df_qubitization")

table = compare_estimates([r1, r2])
for row in table:
    print(f"{row['algorithm_name']:<20} "
          f"logical={row['logical_qubits']} "
          f"physical={row['physical_qubits']} "
          f"T={row['t_count']}")

## 7. One-Call Resource Comparison

The ``molecular_resource_comparison()`` function runs both
Trotter QPE and qubitization QPE and prints a formatted table.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_resource_comparison

# Requires qsharp for resource estimation
# molecular_resource_comparison(
#     "H 0 0 0; H 0 0 0.74",
#     basis="sto-3g",
#     n_estimation_qubits=8,
# )

print("molecular_resource_comparison() prints a table like:")
print()
print("Algorithm               Logical Q    T-gates       Phys Q        Runtime  Code Dist")
print("-" * 83)
print("trotter_qpe                     6          9       150040    2 millisecs         11")
print("qubitization_qpe                9          9       212264   19 millisecs         13")

## 8. Qubit Tapering

Reduce Hamiltonian size by exploiting Z2 symmetries before
running QPE or qubitization.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_hamiltonian
from qdk_pythonic.domains.common.tapering import (
    taper_hamiltonian, find_z2_symmetries,
)

# H2 in 6-31G basis (4 qubits)
h_631g = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", basis="6-31g")
print(f"Before tapering: {h_631g.qubit_count()} qubits, {len(h_631g)} terms")

# Find and apply symmetries
symmetries = find_z2_symmetries(h_631g)
print(f"Z2 symmetries: {len(symmetries)}")
for s in symmetries:
    print(f"  {dict(s.pauli_ops)}")

tapered_h, info = taper_hamiltonian(h_631g)
print(f"
After tapering: {info.tapered_qubits} qubits, {len(tapered_h)} terms")
print(f"Qubits removed: {info.pivot_qubits}")

# The tapered Hamiltonian can now be used for QPE or VQE
# with fewer qubits and lower circuit cost